In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import numpy as np
import pandas as pd
from pathlib import Path

ROOT_PATH = "/kaggle/input/competitions/sjtu-2026spring-cnn/"

class MyDataset(Dataset):
    def __init__(self, x, y):
        self._x = x
        self._y = y

    @staticmethod
    def _to_chw(image: torch.Tensor) -> torch.Tensor:
        if image.ndim == 1:
            if image.numel() == 28 * 28:
                return image.view(1, 28, 28)
            if image.numel() == 32 * 32:
                return image.view(1, 32, 32)
            if image.numel() == 3 * 32 * 32:
                return image.view(3, 32, 32)
            side = int(image.numel() ** 0.5)
            if side * side == image.numel():
                return image.view(1, side, side)
            raise ValueError(f"Unsupported flat image length: {image.numel()}")

        if image.ndim == 2:
            return image.unsqueeze(0)

        if image.ndim == 3 and image.shape[0] not in (1, 3) and image.shape[-1] in (1, 3):
            return image.permute(2, 0, 1)

        return image

    @staticmethod
    def _preprocess(image: torch.Tensor) -> torch.Tensor:
        # 支持单张图像 [C, H, W] 和批量图像 [B, C, H, W]
        if image.ndim == 4:
            return torch.stack([MyDataset._preprocess(item) for item in image], dim=0)

        image = MyDataset._to_chw(image)

        # 像素值统一到 [0, 1]
        if image.max() > 1.0:
            image = image / 255.0

        # 按通道标准化：1通道走CIFAR10统计量，3通道走CIFAR10统计量
        if image.shape[0] == 1:
            mean = torch.tensor([0.1307], dtype=image.dtype, device=image.device).view(-1, 1, 1)
            std = torch.tensor([0.3081], dtype=image.dtype, device=image.device).view(-1, 1, 1)
        else:
            mean = torch.tensor([0.4914, 0.4822, 0.4465], dtype=image.dtype, device=image.device).view(-1, 1, 1)
            std = torch.tensor([0.2470, 0.2435, 0.2616], dtype=image.dtype, device=image.device).view(-1, 1, 1)

        image = (image - mean) / (std + 1e-8)
        return image

    def __getitem__(self, idx):
        image = torch.tensor(self._x[idx], dtype=torch.float32)
        label = torch.tensor(self._y[idx], dtype=torch.long)

        # 先统一成 [C, H, W]
        image = self._to_chw(image)

        image = self._preprocess(image)
        return image, label

    def __len__(self):
        return len(self._x)

def prepare_data_loader(
    path: str,
    ratio: float = None,
    train_batch_size: int = 1,
    is_train: bool = True,
) -> dict:
    """
    参数:
        path (str): .npz格式的数据集文件路径
        ratio (float): 训练集比例
        train_batch_size (int): 批次大小
        num_workers (int): 数据加载的工作进程数
    返回:
        dict: 包含训练和测试数据加载器的字典
    """
    print("开始加载数据...")  # 添加调试信息
    train_dataset = torch.load(path, weights_only = False)
    train_dataset = MyDataset(train_dataset[0], train_dataset[1])
    if is_train:
        train_dataset, val_dataset = torch.utils.data.random_split(train_dataset, [int(len(train_dataset) * ratio), len(train_dataset) - int(len(train_dataset) * ratio)])
        train_loader = torch.utils.data.DataLoader(
                                                    dataset=train_dataset, 
                                                batch_size=train_batch_size, 
                                                shuffle=True,
                                                drop_last=True,
                                                pin_memory=True  
                                                )
        val_loader = torch.utils.data.DataLoader(
                                                dataset=val_dataset, 
                                                batch_size=train_batch_size, 
                                                shuffle=False,
                                                pin_memory=True  
        )
        print("数据加载器创建完成")  # 添加调试信息

        return {"train": train_loader, "val": val_loader}
    else:
        test_loader = torch.utils.data.DataLoader(
                                                dataset=train_dataset, 
                                                batch_size=train_batch_size, 
                                                shuffle=False,
                                                pin_memory=True  
        )
        print("测试数据加载器创建完成")  # 添加调试信息

        return {"test": test_loader}


In [ ]:
class SimpleCNN(nn.Module):
    """
    可配置的CNN模型
    
    参数:
        in_channels (int): 输入通道数
        num_classes (int): 分类类别数
        conv_layers (list): 每个卷积层的输出通道数列表
        fc_layers (list): 每个全连接层的输出维度列表
        kernel_size (int): 卷积核大小
        dropout_rate (float): Dropout比率
    """
    def __init__(
        self,
        in_channels: int = 3,
        num_classes: int = 10,
        conv_layers: list = [32, 64],  # 默认两层卷积
        fc_layers: list = [128, 64],   # 默认两层全连接
        kernel_size: int = 3,
        dropout_rate: float = 0.1
    ):
        super().__init__()
        
        # 构建卷积层
        self.conv_blocks = nn.ModuleList()
        current_channels = in_channels
        
        for i, out_channels in enumerate(conv_layers):
            conv_block = nn.Sequential(
                nn.Conv2d(
                    in_channels=current_channels,
                    out_channels=out_channels,
                    kernel_size=kernel_size,
                    stride=1,
                    padding='same'
                ),
                # nn.BatchNorm2d(out_channels), 可选择是否添加BatchNorm层
                nn.ReLU(),
                nn.MaxPool2d(kernel_size=2)
            )
            self.conv_blocks.append(conv_block)
            current_channels = out_channels
           
            
        # 计算展平后的特征维度
        # 每经过一次MaxPool2d或AvgPool2d，特征图尺寸减半
        feature_size = current_channels * (32 // (2 ** len(conv_layers))) ** 2
        
        # 构建全连接层
        self.fc_blocks = nn.ModuleList()
        current_dim = feature_size
        
        for fc_dim in fc_layers:
            fc_block = nn.Sequential(
                nn.Linear(current_dim, fc_dim),
                nn.ReLU(),
                nn.Dropout(dropout_rate)
            )
            self.fc_blocks.append(fc_block)
            current_dim = fc_dim
            
        # 输出层
        self.output_layer = nn.Linear(current_dim, num_classes)

    def forward(self, x):
         # 通过所有卷积层
         for conv_block in self.conv_blocks:
             x = conv_block(x)
    
         # 展平
         x = torch.flatten(x, start_dim=1)
    
         # 通过所有全连接层
         for fc_block in self.fc_blocks:
              x = fc_block(x)
        
         # 输出层
         x = self.output_layer(x)
         return x 
        


In [ ]:
import torch.optim as optim
import matplotlib.pyplot as plt
import time 

loss_fn = nn.CrossEntropyLoss()
time0 = time.time()
def train_step(
    model: nn.Module, optimizer, batch: dict, device: torch.device
):
    """
    单步训练
    """
    batch_images, labels = batch
    batch_images = batch_images.to(device)
    labels = labels.to(device)
        
    optimizer.zero_grad()

    logits = model(batch_images) # 模型正向过程
        
    loss = loss_fn(logits, labels) # 计算总损失
        
    loss.backward() # 反向传播
        
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) # 添加梯度裁剪，防止梯度爆炸
        
    optimizer.step()

    return loss.item(), logits, labels
        

def eval_step(model: nn.Module, batch: dict, device: torch.device):
    # 单步评估
    model.eval()
    with torch.no_grad():
        batch_images, labels = batch
        batch_images = batch_images.to(device)
        labels = labels.to(device) 
        logits = model(batch_images) # 模型正向过程
        loss = loss_fn(logits, labels) # 计算总损失
        return loss.item(), logits, labels
    
def train_per_epoch(
    model: nn.Module,
    optimizer: optim.Optimizer,
    batch_size: int,
    train_loader: DataLoader,
    device: torch.device,
    global_batch_offset: int,
):
    model.train()
    num_data = len(train_loader.dataset)
    num_batches = len(train_loader)
    correct = 0
    total_loss = 0
    print(f"开始训练 - 总样本数: {num_data}")
    print(f"总批次数: {len(train_loader)}")

    train_loss_list = []
    train_acc_list = []

    test_loss_list = []
    test_acc_list = []
    test_batch_idx = []
    for batch_idx, batch in enumerate(train_loader):
        loss, logits, labels = train_step(model, optimizer, batch, device)
        total_loss += loss
        
        # 计算当前批次的预测值
        _, predicted = torch.max(logits, 1)
        correct += (predicted == labels).sum().item()
        
        # ✅ 计算当前批次的准确率
        batch_correct = (predicted == labels).sum().item()
        batch_total = len(labels)
        batch_acc = batch_correct / batch_total  # 当前批次的准确率
    
        # 每5个批次打印一次
        if batch_idx % 5 == 0:
            current = batch_idx * batch_size + len(batch[0])
            print(f"批次 {batch_idx} Train: Loss: {loss:>6.4f}, Acc: {100*batch_acc:.4f}%, 进度: {current:>5d}/{num_data:>5d}")            
            print(f"批次 {batch_idx}: Finished in {time.time() - time0:.2f} seconds")
            
        if batch_idx % 15 == 0:
            test_loss, test_acc = val_per_epoch(model, val_loader, device)
            test_loss_list.append(test_loss) 
            test_acc_list.append(test_acc)
            test_batch_idx.append(global_batch_offset + batch_idx)
            print(f"批次 {batch_idx} Val : Loss: {test_loss:>6.4f}, Acc: {100*test_acc:.4f}%")
            print(f"批次 {batch_idx}: Finished in {time.time() - time0:.2f} seconds")
        
        # ✅ 记录每个批次的 loss 和 acc
        train_loss_list.append(loss) 
        train_acc_list.append(batch_acc)
        
    test_loss, test_acc = val_per_epoch(model, val_loader, device)
    test_loss_list.append(test_loss) 
    test_acc_list.append(test_acc)
    test_batch_idx.append(global_batch_offset + batch_idx)
    print(f"批次 {batch_idx} Validation : Loss: {test_loss:>6.4f}, Acc: {100*test_acc:.4f}%")
    print(f"批次 {batch_idx}: Finished in {time.time() - time0:.2f} seconds")
    accuracy = correct / num_data
    avg_loss = total_loss / num_batches
    print(f"Train Error: \n Accuracy: {(100*accuracy):>0.1f}%, Avg loss: {avg_loss:>.8f} \n")
    return train_loss_list, train_acc_list, test_loss_list, test_acc_list, test_batch_idx
        
def val_per_epoch(
    model: nn.Module,
    val_loader: DataLoader,
    device: torch.device,
):
    """
    每轮测试
    """
    model.eval()
    total_loss = 0.0
    correct = 0
    num_batches = len(val_loader)
    num_data = len(val_loader.dataset)
    with torch.no_grad():
        for batch in val_loader:
            loss, logits, labels = eval_step(model, batch, device)
            total_loss += loss
            _, predicted = torch.max(logits, 1)
            correct += (predicted == labels).sum().item()

    avg_loss = total_loss / num_batches
    accuracy = correct / num_data
    return avg_loss, accuracy

def draw_loss_accuracy_curves(train_losses, train_acc_list, test_batch_idx_list, test_loss_list, test_acc_list):
    plt.figure(figsize=(18, 5))

    # 绘制 loss 曲线
    plt.subplot(1, 3, 1)
    plt.plot(train_losses, label='Train Loss', alpha=0.5)
    plt.plot(test_batch_idx_list, test_loss_list, label='Validation Loss', marker='o', markersize=3)
    plt.title('Loss vs Batch Index')
    plt.xlabel('Batch Index')
    plt.ylabel('Loss')
    plt.legend()

    # 绘制 train acc 和 val acc
    plt.subplot(1, 2, 2)
    plt.plot(train_acc_list, label='Train Acc', alpha=0.5)
    plt.plot(test_batch_idx_list, test_acc_list, label='Validation Acc', marker='o', markersize=3)
    plt.title('Accuracy vs Batch Index')
    plt.xlabel('Batch Index')
    plt.ylabel('Accuracy')
    plt.legend()

    # # 绘制 validation accuracy 单独
    # plt.subplot(1, 3, 3)
    # plt.plot(test_batch_idx_list, test_acc_list, label='Validation Accuracy', marker='o', markersize=3)
    # plt.title('Validation Accuracy vs Batch Index')
    # plt.xlabel('Batch Index')
    # plt.ylabel('Accuracy')
    # plt.legend()

    plt.tight_layout()
    plt.show()


def controller(
    train_loader,
    val_loader,
    seed: int,
    # CNN特有参数
    in_channels: int = None,
    conv_layers: list = None,
    kernel_size: int = None,
    fc_layers  : list = None,
    # 通用参数
    num_classes: int = 10,
    dropout_rate: float = 0.1,
    ratio: float = 0.8,
    train_batch_size: int = 64,
    num_workers: int = 4,
    epochs: int = 10,
    learning_rate: float = 0.001,
    weight_decay: float = 0.004
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    torch.manual_seed(seed)
    model = SimpleCNN(
            in_channels=in_channels,
            num_classes=num_classes,
            conv_layers=conv_layers,
            fc_layers=fc_layers,  
            kernel_size=kernel_size,
            dropout_rate=dropout_rate
        ).to(device)

    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    # print(f"Total Epoch: {epochs}")
    best_acc = 0.0
    train_loss_list = []
    train_acc_list = []
    test_loss_list = []
    test_acc_list = []
    test_batch_idx_list = []

    # 初始 validation（对应 batch index 0）
    test_loss, test_acc = val_per_epoch(model, val_loader, device)
    test_loss_list.append(test_loss)
    test_acc_list.append(test_acc)
    test_batch_idx_list.append(0)

    num_batches_per_epoch = len(train_loader)

    for epoch in range(epochs):
        print(f"Epoch {epoch + 1}/{epochs} strated in {time.time() - time0:.2f} seconds")
        global_batch_offset = epoch * num_batches_per_epoch

        train_loss, train_acc, test_loss, test_acc, test_batch_idx = train_per_epoch(
            model, optimizer, train_batch_size, train_loader, device, global_batch_offset
        )
        scheduler.step()
        if test_acc[-1] > best_acc:
            best_acc = test_acc[-1]
            torch.save(model.state_dict(), 'best_model.pth')
        train_loss_list.extend(train_loss)
        train_acc_list.extend(train_acc)
        test_loss_list.extend(test_loss)
        test_acc_list.extend(test_acc)
        test_batch_idx_list.extend(test_batch_idx)
        
        print(f"Epoch {epoch + 1} Finished in {time.time() - time0:.2f} seconds")
        print(f"Validation accuracy: {100*test_acc[-1]:.2f}%")
        draw_loss_accuracy_curves(train_loss_list, train_acc_list, test_batch_idx_list, test_loss_list, test_acc_list)
        
    print(f"Training completed! Best accuracy: {100*best_acc:.2f}%")
    
    return model

def show_sample_images(sample, title):
    images, labels = sample
    images = images.cpu().numpy()
    labels = labels.cpu().numpy()
    
    plt.figure(figsize=(8, 8))
    for i in range(9):
        plt.subplot(3, 3, i + 1)
        if images.shape[1] == 1:  # 灰度图像
            plt.imshow(images[i][0], cmap='gray')
        else:  # 彩色图像
            img = np.transpose(images[i], (1, 2, 0))  # 转换为HWC格式
            img = (img * np.array([0.2470, 0.2435, 0.2616]) + np.array([0.4914, 0.4822, 0.4465])) * 255.0  # 反标准化
            img = np.clip(img.astype(np.uint8), 0, 255)  # 确保像素值在有效范围内
            plt.imshow(img)
        plt.title(f"Label: {labels[i]}")
        plt.axis('off')
    plt.suptitle(title)
    plt.show()

In [ ]:
# 随机数种子：确保每次运行代码得到相同的结果
# 请同学把随机种子改成自己的学号，方便助教后期验证结果的真实性。
seed = 【这里改成学号后八位】
# 例如：
# seed = 12345678
# activation 是一个字符串，指定了隐藏层使用的激活函数类型。常见的激活函数包括 'relu'（修正线性单元）、'tanh'（双曲正切）和 'sigmoid'（S形函数）。例如，activation='relu' 表示使用ReLU激活函数。
activation = 'relu'
# num_classes 是一个整数，表示分类任务中的类别数量。例如，对于CIFAR10数据集，num_classes=10，因为CIFAR10包含10个不同的类别。
num_classes = 10
# dropout_rate 是一个浮点数，表示在训练过程中应用的Dropout正则化的比率。Dropout是一种防止过拟合的技术，通过在训练过程中随机丢弃神经元来实现。例如，dropout_rate=0.1 表示每个神经元有10%的概率被丢弃。
dropout_rate = 0.2
# train_batch_size 是一个整数，表示在训练过程中每个批次中包含的样本数量。较大的批次大小可以加速训练，但可能需要更多的内存。例如，train_batch_size=64 表示每个批次包含64个样本。
train_batch_size: int = 1024
# epochs 是一个整数，表示整个训练数据集要被训练的次数。每个epoch表示模型已经看过整个训练数据集一次。例如，epochs=10 表示模型将对整个训练数据集进行10次迭代。
epochs: int = 1
# learning_rate 是一个浮点数，表示模型在训练过程中调整参数的步长。较大的学习率可能导致训练不稳定，而较小的学习率可能导致训练过慢。例如，
learning_rate: float = 0.001
# weight_decay 是一个浮点数，表示L2正则化的强度。L2正则化通过在损失函数中添加模型参数的平方和来防止过拟合。例如，
weight_decay: float = 0.004
# in_channels 是一个整数，表示输入图像的通道数。例如，对于CIFAR10数据集，in_channels=3
in_channels: int = 3
# conv_layers 是一个列表，定义了每个卷积层的输出通道数。例如，conv_layers=[32, 64] 表示有两个卷积层，第一层输出32个通道，第二层输出64个通道。
conv_layers: list = [32, 64]
# conv_layers: list = [4, 4]
# kernel_size 是一个整数，表示卷积核的大小。例如，kernel_size=3
kernel_size: int = 3
# fc_layers 是一个列表，定义了每个全连接层的输出维度。例如，fc_layers=[128, 64] 表示有两个全连接层，第一层输出128维，第二层输出64维。
fc_layers: list = [64, 64]
# fc_layers: list = [5, 5]

#ratio比率：训练集中用于训练的数据量/总数据量
#比率0.8表示50000张图片中有40000张用于训练，10000用于测试
ratio = 0.8

loader_dict = prepare_data_loader(path=Path(f"{ROOT_PATH}/training.pt"), ratio=ratio, train_batch_size=train_batch_size,)
train_loader = loader_dict["train"]
val_loader = loader_dict["val"]
# 增加一个 train 和 val 的图像示意图
train_sample = next(iter(train_loader))
val_sample = next(iter(val_loader))

show_sample_images(train_sample, "Train Sample Images")
show_sample_images(val_sample, "Validation Sample Images")
# print(f"Start Total Epoch: {epochs}")
model = controller(
    train_loader=train_loader,
    val_loader=val_loader,
    seed=seed,
    in_channels=in_channels,
    conv_layers=conv_layers,
    kernel_size=kernel_size,
    fc_layers=fc_layers,
    num_classes=num_classes,
    dropout_rate=dropout_rate,
    ratio = ratio,
    train_batch_size=train_batch_size,
    epochs=epochs,  
    learning_rate=learning_rate,
    weight_decay = weight_decay
)


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def evaluater_with_dataloader_and_save(model, test_path, device, solution_path=None):
    """
    使用 val_dataloader 计算准确率并可选择保存预测结果
    
    参数:
    - model: 需要评估的模型
    - val_loader: 测试数据加载器
    - device: 计算设备 (CPU 或 GPU)
    - solution_path: 保存预测结果的路径(可选)
    
    返回:
    - accuracy: 模型在测试集上的准确率
    """
    model.eval()  # 设置为评估模式
    all_predictions = []
    test_dataset = torch.load(test_path, weights_only = False)
    
    test_dataset = MyDataset(test_dataset, torch.ones(len(test_dataset), dtype=torch.long))  # 创建一个假的标签数组，实际不会使用
    test_dataloader = torch.utils.data.DataLoader(test_dataset, batch_size=128, shuffle=False, pin_memory=True)
    all_ids = []
    with torch.no_grad():
        for batch_idx, (images, _) in enumerate(test_dataloader):
            images = images.to(device)
            logits = model(images)
            predictions = torch.argmax(logits, dim=1).cpu().numpy()
            all_predictions.extend(predictions)
            all_ids.extend(range(batch_idx * 128, batch_idx * 128 + len(predictions)))
    
    # 保存预测结果
    if solution_path is not None:
        predictions_df = pd.DataFrame({"ID": all_ids, "label": all_predictions})
        predictions_df.to_csv(solution_path, index=False)
        print(f'预测结果已保存至 {solution_path}')

# test_dataloader = prepare_data_loader(path=Path(f"{ROOT_PATH}/test-shuffled-nolabel.pt"),train_batch_size=128, is_train=False)['test']
evaluater_with_dataloader_and_save(
    model,
    test_path=Path(f"{ROOT_PATH}/test-shuffled-nolabel.pt"),
    solution_path=Path("submission.csv"),
    device = device
    )
